In [1]:
import os, sys
import pandas as pd
import numpy as np

cwd = os.getcwd()

file = 'FR001400AJ45_20231009_LOB.parquet.gzip'
LOB = pd.read_parquet(file)

In [2]:
df = LOB.copy()
df = df.between_time('9:00', '17:35')
display(df.loc['2023-10-09 09:00:00'])

,price,side,size
index,,,
2023-10-09 09:00:00,14.10,Buy,8.0
2023-10-09 09:00:00,15.00,Buy,10.0
2023-10-09 09:00:00,15.10,Buy,400.0
2023-10-09 09:00:00,17.00,Buy,300.0
2023-10-09 09:00:00,17.60,Buy,20.0
...,...,...,...
2023-10-09 09:00:00,40.00,Sell,650.0
2023-10-09 09:00:00,40.25,Sell,12.0
2023-10-09 09:00:00,40.75,Sell,124.0


In [3]:
t = df.loc[(df['side'] == 'Buy') & (df.index <= '2023-10-09 09:05:00')]

tick_p = 0.01
w_int = tick_p * 10
bins = [t['price'].max()-v*w_int for v in range(60+1)]

t['interval'] = pd.cut(t['price'], bins=bins[::-1], right=True)

display(t.sort_values('price', ascending=False)[:15], t[t['price'] > 28.95]['size'].sum())

t['price'] = t['interval'].apply(lambda x: x.left)

t = t.reset_index()

t = t.groupby(['index','side','price'])['size'].sum().reset_index()
t = t.sort_values('price', ascending=False).reset_index()
t['rank'] = t.groupby('index')['price'].rank(ascending=False)
t['idx'] = 0
display(t)

<ipython-input-3-5392c2f44b28>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  t['interval'] = pd.cut(t['price'], bins=bins[::-1], right=True)


,price,side,size,interval
index,,,,
2023-10-09 09:00:00,29.05,Buy,520.0,"(28.95, 29.05]"
2023-10-09 09:00:00,29.04,Buy,2896.0,"(28.95, 29.05]"
2023-10-09 09:00:00,29.03,Buy,3519.0,"(28.95, 29.05]"
2023-10-09 09:00:00,29.02,Buy,1732.0,"(28.95, 29.05]"
2023-10-09 09:00:00,29.01,Buy,3326.0,"(28.95, 29.05]"
2023-10-09 09:00:00,29.00,Buy,2043.0,"(28.95, 29.05]"
2023-10-09 09:00:00,28.99,Buy,3223.0,"(28.95, 29.05]"
2023-10-09 09:00:00,28.98,Buy,3899.0,"(28.95, 29.05]"
2023-10-09 09:00:00,28.97,Buy,2212.0,"(28.95, 29.05]"


35684.0

<ipython-input-3-5392c2f44b28>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  t['price'] = t['interval'].apply(lambda x: x.left)


,level_0,index,side,price,size,rank,idx
0,119,2023-10-09 09:05:00,Buy,28.95,5491.0,1.0,0
1,59,2023-10-09 09:00:00,Buy,28.95,30193.0,1.0,0
2,58,2023-10-09 09:00:00,Buy,28.85,14232.0,2.0,0
3,118,2023-10-09 09:05:00,Buy,28.85,24590.0,2.0,0
4,57,2023-10-09 09:00:00,Buy,28.75,3730.0,3.0,0
...,...,...,...,...,...,...,...
115,62,2023-10-09 09:05:00,Buy,23.25,70.0,58.0,0
116,61,2023-10-09 09:05:00,Buy,23.15,5.0,59.0,0
117,1,2023-10-09 09:00:00,Buy,23.15,5.0,59.0,0
118,0,2023-10-09 09:00:00,Buy,23.05,0.0,60.0,0


In [4]:
test = t.pivot(index=['index'], columns=['rank','side'], values=['price', 'size']).T.reset_index()
display(test)
test = test.groupby(['rank','side','level_0']).last()
display(test.T, test)

index,level_0,rank,side,2023-10-09 09:00:00,2023-10-09 09:05:00
0,price,1.0,Buy,28.95,28.95
1,price,2.0,Buy,28.85,28.85
2,price,3.0,Buy,28.75,28.75
3,price,4.0,Buy,28.65,28.65
4,price,5.0,Buy,28.55,28.55
...,...,...,...,...,...
115,size,56.0,Buy,124.00,124.00
116,size,57.0,Buy,40.00,40.00
117,size,58.0,Buy,70.00,70.00
118,size,59.0,Buy,5.00,5.00


rank                  1.0             2.0             3.0            4.0   \
side                   Buy             Buy             Buy            Buy   
level_0              price     size  price     size  price    size  price   
index                                                                       
2023-10-09 09:00:00  28.95  30193.0  28.85  14232.0  28.75  3730.0  28.65   
2023-10-09 09:05:00  28.95   5491.0  28.85  24590.0  28.75  6806.0  28.65   

rank                          5.0           ...   56.0          57.0        \
side                           Buy          ...    Buy           Buy         
level_0                size  price    size  ...  price   size  price  size   
index                                       ...                              
2023-10-09 09:00:00  9988.0  28.55  1391.0  ...  23.45  124.0  23.35  40.0   
2023-10-09 09:05:00  9592.0  28.55  1391.0  ...  23.45  124.0  23.35  40.0   

rank                  58.0         59.0        60.0       
side                   Buy          Buy         Buy       
level_0              price  size  price size  price size  
index                                                     
2023-10-09 09:00:00  23.25  70.0  23.15  5.0  23.05  0.0  
2023-10-09 09:05:00  23.25  70.0  23.15  5.0  23.05  0.0  

[2 rows x 120 columns]

index              2023-10-09 09:00:00  2023-10-09 09:05:00
rank side level_0                                          
1.0  Buy  price                  28.95                28.95
          size                30193.00              5491.00
2.0  Buy  price                  28.85                28.85
          size                14232.00             24590.00
3.0  Buy  price                  28.75                28.75
...                                ...                  ...
58.0 Buy  size                   70.00                70.00
59.0 Buy  price                  23.15                23.15
          size                    5.00                 5.00
60.0 Buy  price                  23.05                23.05
          size                    0.00                 0.00

[120 rows x 2 columns]

In [15]:
t = df.loc[(df['side'] == 'Buy') & (df.index <= '2023-10-09 09:05:00')]

final_b = pd.DataFrame()
ls_b = []

for _, chunk in t.groupby('index'):

    tick_p = 0.01
    w_int = tick_p * 10
    bins = [chunk['price'].max()-v*w_int for v in range(60+1)]

    chunk['interval'] = pd.cut(chunk['price'], bins=bins[::-1], right=True)

    chunk['price'] = chunk['interval'].apply(lambda x: x.left)

    chunk = chunk.reset_index()

    chunk = chunk.groupby(['index','side','price'])['size'].sum().reset_index()
    chunk = chunk.sort_values('price', ascending=False).reset_index()
    chunk['rank'] = chunk.groupby('index')['price'].rank(ascending=False)
    chunk['idx'] = 0
    
    test = chunk.pivot(index=['index'], columns=['rank','side'], values=['price', 'size']).T.reset_index()
    test = test.groupby(['rank','side','level_0']).last()
    final_b = pd.concat([final_b, test.T])
    ls_b.append(chunk['size'].sum())
    
display(final_b)

rank                  1.0             2.0             3.0            4.0   \
side                   Buy             Buy             Buy            Buy   
level_0              price     size  price     size  price    size  price   
index                                                                       
2023-10-09 09:00:00  28.95  30193.0  28.85  14232.0  28.75  3730.0  28.65   
2023-10-09 09:05:00  28.87  28877.0  28.77   7861.0  28.67  9668.0  28.57   

rank                          5.0           ...   56.0          57.0        \
side                           Buy          ...    Buy           Buy         
level_0                size  price    size  ...  price   size  price  size   
index                                       ...                              
2023-10-09 09:00:00  9988.0  28.55  1391.0  ...  23.45  124.0  23.35  40.0   
2023-10-09 09:05:00  1391.0  28.47  2264.0  ...  23.37   40.0  23.27  70.0   

rank                  58.0         59.0        60.0          
side                   Buy          Buy         Buy          
level_0              price  size  price size  price    size  
index                                                        
2023-10-09 09:00:00  23.25  70.0  23.15  5.0  23.05     0.0  
2023-10-09 09:05:00  23.17   5.0  23.07  0.0  22.97  4479.0  

[2 rows x 120 columns]

In [16]:
t = df.loc[(df['side'] == 'Sell') & (df.index <= '2023-10-09 09:05:00')]

final_s = pd.DataFrame()
ls_s = []
display(t[:11])
for _, chunk in t.groupby('index'):

    tick_p = 0.01
    w_int = tick_p * 10
    bins = [chunk['price'].min()+v*w_int for v in range(60+1)]

    chunk['interval'] = pd.cut(chunk['price'], bins=bins, right=False)

    chunk['price'] = chunk['interval'].apply(lambda x: x.right)

    chunk = chunk.reset_index()

    chunk = chunk.groupby(['index','side','price'])['size'].sum().reset_index()
    chunk = chunk.sort_values('price', ascending=True).reset_index()
    chunk['rank'] = chunk.groupby('index')['price'].rank(ascending=True)
    chunk['idx'] = 0
    
    test = chunk.pivot(index=['index'], columns=['rank','side'], values=['price', 'size']).T.reset_index()
    test = test.groupby(['rank','side','level_0']).last()
    final_s = pd.concat([final_s, test.T])
    ls_s.append(chunk['size'].sum())
    
display(final_s)

,price,side,size
index,,,
2023-10-09 09:00:00,29.06,Sell,169.0
2023-10-09 09:00:00,29.07,Sell,4354.0
2023-10-09 09:00:00,29.08,Sell,1619.0
2023-10-09 09:00:00,29.09,Sell,2288.0
2023-10-09 09:00:00,29.10,Sell,5651.0
2023-10-09 09:00:00,29.11,Sell,5136.0
2023-10-09 09:00:00,29.12,Sell,1414.0
2023-10-09 09:00:00,29.13,Sell,3120.0
2023-10-09 09:00:00,29.14,Sell,1662.0


rank                  1.0             2.0             3.0            4.0   \
side                  Sell            Sell            Sell           Sell   
level_0              price     size  price     size  price    size  price   
index                                                                       
2023-10-09 09:00:00  29.16  26828.0  29.26  13244.0  29.36  1640.0  29.46   
2023-10-09 09:05:00  29.08  35458.0  29.18  14784.0  29.28  7045.0  29.38   

rank                          5.0           ...   56.0          57.0         \
side                          Sell          ...   Sell          Sell          
level_0                size  price    size  ...  price   size  price   size   
index                                       ...                               
2023-10-09 09:00:00  6515.0  29.56  2164.0  ...  34.66   30.0  34.76  591.0   
2023-10-09 09:05:00  6763.0  29.48  1383.0  ...  34.58  270.0  34.68   30.0   

rank                  58.0          59.0         60.0          
side                  Sell          Sell         Sell          
level_0              price   size  price  size  price    size  
index                                                          
2023-10-09 09:00:00  34.86    0.0  34.96  57.0  35.06  3011.0  
2023-10-09 09:05:00  34.78  591.0  34.88   0.0  34.98    95.0  

[2 rows x 120 columns]

In [17]:
final = pd.concat([final_b.T, final_s.T], axis=0)
final = final.groupby(['rank','side','level_0']).last().T
display(final)

rank                  1.0                             2.0                   \
side                   Buy            Sell             Buy            Sell   
level_0              price     size  price     size  price     size  price   
index                                                                        
2023-10-09 09:00:00  28.95  30193.0  29.16  26828.0  28.85  14232.0  29.26   
2023-10-09 09:05:00  28.87  28877.0  29.08  35458.0  28.77   7861.0  29.18   

rank                           3.0           ...   58.0          59.0       \
side                            Buy          ...   Sell           Buy        
level_0                 size  price    size  ...  price   size  price size   
index                                        ...                             
2023-10-09 09:00:00  13244.0  28.75  3730.0  ...  34.86    0.0  23.15  5.0   
2023-10-09 09:05:00  14784.0  28.67  9668.0  ...  34.78  591.0  23.07  0.0   

rank                               60.0                         
side                  Sell          Buy           Sell          
level_0              price  size  price    size  price    size  
index                                                           
2023-10-09 09:00:00  34.96  57.0  23.05     0.0  35.06  3011.0  
2023-10-09 09:05:00  34.88   0.0  22.97  4479.0  34.98    95.0  

[2 rows x 240 columns]

In [18]:
display(final.columns)

MultiIndex([( 1.0,  'Buy', 'price'),
            ( 1.0,  'Buy',  'size'),
            ( 1.0, 'Sell', 'price'),
            ( 1.0, 'Sell',  'size'),
            ( 2.0,  'Buy', 'price'),
            ( 2.0,  'Buy',  'size'),
            ( 2.0, 'Sell', 'price'),
            ( 2.0, 'Sell',  'size'),
            ( 3.0,  'Buy', 'price'),
            ( 3.0,  'Buy',  'size'),
            ...
            (58.0, 'Sell', 'price'),
            (58.0, 'Sell',  'size'),
            (59.0,  'Buy', 'price'),
            (59.0,  'Buy',  'size'),
            (59.0, 'Sell', 'price'),
            (59.0, 'Sell',  'size'),
            (60.0,  'Buy', 'price'),
            (60.0,  'Buy',  'size'),
            (60.0, 'Sell', 'price'),
            (60.0, 'Sell',  'size')],
           names=['rank', 'side', 'level_0'], length=240)

In [19]:
final.columns = ['{}_{}_{}'.format(side.lower(), key, int(rank)) for rank, side, key in final.columns]
final['buy_liquidity'] = ls_b
final['sell_liquidity'] = ls_s
display(final)

,buy_price_1,buy_size_1,sell_price_1,sell_size_1,buy_price_2,buy_size_2,sell_price_2,sell_size_2,buy_price_3,buy_size_3,...,buy_price_59,buy_size_59,sell_price_59,sell_size_59,buy_price_60,buy_size_60,sell_price_60,sell_size_60,buy_liquidity,sell_liquidity
index,,,,,,,,,,,,,,,,,,,,,
2023-10-09 09:00:00,28.95,30193.0,29.16,26828.0,28.85,14232.0,29.26,13244.0,28.75,3730.0,...,23.15,5.0,34.96,57.0,23.05,0.0,35.06,3011.0,148431.0,192277.0
2023-10-09 09:05:00,28.87,28877.0,29.08,35458.0,28.77,7861.0,29.18,14784.0,28.67,9668.0,...,23.07,0.0,34.88,0.0,22.97,4479.0,34.98,95.0,141186.0,205872.0


In [22]:
final['total_liquidity'] = final[['buy_liquidity','sell_liquidity']].sum(axis=1)
display(final)

,buy_price_1,buy_size_1,sell_price_1,sell_size_1,buy_price_2,buy_size_2,sell_price_2,sell_size_2,buy_price_3,buy_size_3,...,buy_size_59,sell_price_59,sell_size_59,buy_price_60,buy_size_60,sell_price_60,sell_size_60,buy_liquidity,sell_liquidity,total_liquidity
index,,,,,,,,,,,,,,,,,,,,,
2023-10-09 09:00:00,28.95,30193.0,29.16,26828.0,28.85,14232.0,29.26,13244.0,28.75,3730.0,...,5.0,34.96,57.0,23.05,0.0,35.06,3011.0,148431.0,192277.0,340708.0
2023-10-09 09:05:00,28.87,28877.0,29.08,35458.0,28.77,7861.0,29.18,14784.0,28.67,9668.0,...,0.0,34.88,0.0,22.97,4479.0,34.98,95.0,141186.0,205872.0,347058.0
